In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA L4, 23034 MiB


it's Google's latest generation GPU optimized specifically for deep learning inference and training.
23GB VRAM

In [6]:
!gcloud auth login --no-launch-browser


You are running on a Google Compute Engine virtual machine.
It is recommended that you use service accounts for authentication.

You can run:

  $ gcloud config set account `ACCOUNT`

to switch accounts if necessary.

Your credentials may be visible to others with access to this
virtual machine. Are you sure you want to authenticate with
your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.

In [7]:
# Check environment
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Check GCS access
!gsutil ls gs://mywaymo-perdataset-2026/
print("GCS connected ✅")

NVIDIA L4, 23034 MiB
gs://mywaymo-perdataset-2026/samples.npy
gs://mywaymo-perdataset-2026/extracted/
gs://mywaymo-perdataset-2026/models/
gs://mywaymo-perdataset-2026/training/
GCS connected ✅


In [4]:
import os
import io
import tensorflow as tf
import numpy as np
from PIL import Image
from pathlib import Path
import pandas as pd

In [1]:
!pip install protobuf==3.20.3 -q
!pip install waymo-open-dataset-tf-2-11-0 --no-deps -q
print("Installation complete — restart runtime if this is your first run.")


Installation complete — restart runtime if this is your first run.


In [2]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

from waymo_open_dataset import dataset_pb2
print("Waymo proto available ✅")

Waymo proto available ✅


In [5]:
# Authenticate
!gcloud config set project solar-cycle-487619-t6
!gsutil ls gs://mywaymo-perdataset-2026/

# Config
GCS_TRAINING  = "gs://mywaymo-perdataset-2026/training"
GCS_EXTRACTED = "gs://mywaymo-perdataset-2026/extracted"
LOCAL_WORK    = Path("/content/extraction")
LOCAL_WORK.mkdir(exist_ok=True)

# Get all TFRecords
result = !gsutil ls {GCS_TRAINING}/
tfrecords = [r.strip() for r in result if '.tfrecord' in r]
print(f"Total segments available: {len(tfrecords)}")
tfrecords_50 = tfrecords[:50]
print(f"First 50 selected for extraction")

Project 'solar-cycle-487619-t6' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].
gs://mywaymo-perdataset-2026/samples.npy
gs://mywaymo-perdataset-2026/extracted/
gs://mywaymo-perdataset-2026/models/
gs://mywaymo-perdataset-2026/training/
Total segments available: 798
First 50 selected for extraction


In [9]:
#extraction pipeline:

def extract_segment(tfrecord_path, output_gcs_path):
    """Extract all frames from one segment and save to GCS."""

    seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord', '').replace('.tfrecord-00000-of-00001', '')
    local_seg_dir = LOCAL_WORK / seg_name
    local_seg_dir.mkdir(exist_ok=True)

    print(f"\nExtracting: {seg_name[:50]}...")

    # Download TFRecord
    local_tfrecord = LOCAL_WORK / "current.tfrecord"
    !gsutil cp "{tfrecord_path}" "{local_tfrecord}" -q

    # Parse frames
    dataset = tf.data.TFRecordDataset(str(local_tfrecord))
    labels_data = []
    frame_count = 0

    camera_names = {1: 'FRONT', 2: 'FRONT_LEFT', 3: 'FRONT_RIGHT', 4: 'SIDE_LEFT', 5: 'SIDE_RIGHT'}

    for frame_idx, data in enumerate(dataset):
        frame = dataset_pb2.Frame()
        frame.ParseFromString(data.numpy())

        for camera_image in frame.images:
            cam_name = camera_names.get(camera_image.name, None)
            if cam_name != 'FRONT':
                continue

            # Save image
            img = Image.open(io.BytesIO(camera_image.image))
            img_filename = f"frame_{frame_idx:03d}_FRONT.jpg"
            img_path = local_seg_dir / img_filename
            img.save(str(img_path), 'JPEG')

            # Extract labels
            for camera_label in frame.camera_labels:
                if camera_label.name != camera_image.name:
                    continue
                for label in camera_label.labels:
                    labels_data.append({
                        'frame': frame_idx,
                        'camera': cam_name,
                        'type': label.type,
                        'center_x': label.box.center_x,
                        'center_y': label.box.center_y,
                        'width': label.box.width,
                        'height': label.box.length
                    })

        frame_count += 1

    # Save labels CSV
    labels_df = pd.DataFrame(labels_data)
    csv_path = local_seg_dir / "labels.csv"
    labels_df.to_csv(str(csv_path), index=False)

    # Upload to GCS
    gcs_seg_path = f"{output_gcs_path}/{seg_name}"
    !gsutil -m cp "{local_seg_dir}/*.jpg" "{gcs_seg_path}/" -q
    !gsutil cp "{csv_path}" "{gcs_seg_path}/labels.csv" -q

    # Cleanup local files
    import shutil
    shutil.rmtree(str(local_seg_dir))
    os.remove(str(local_tfrecord))

    print(f"  ✅ {frame_count} frames, {len(labels_df)} labels → {gcs_seg_path}")
    return frame_count, len(labels_df)

print("Extraction function ready ✅")

Extraction function ready ✅


seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord-00000-of-00001', '').replace('.tfrecord', '')

gs://mywaymo-perdataset-2026/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord

Step 1 — .split('/')[-1]
Split by / and take the last piece:

segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord

Step 2 — .replace('.tfrecord-00000-of-00001', '')
Some Waymo files end with .tfrecord-00000-of-00001 — remove that if present:

segment-10017090168044687777_6380_000_6400_000_with_camera_labels

Step 3 — .replace('.tfrecord', '')
Remove the .tfrecord extension:

segment-10017090168044687777_6380_000_6400_000_with_camera_labels

Final result — just the clean segment name:

segment-10017090168044687777_6380_000_6400_000_with_camera_labels

This becomes the folder name in GCS where images are saved.

In [20]:
#beginner friendly extraction pipeline of above cell

def extract_one_segment(tfrecord_path):
    """Download one segment, extract FRONT camera images and labels, upload to GCS."""

    # Get segment name
    seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord-00000-of-00001', '').replace('.tfrecord', '')
    local_dir = f"/content/extracted/{seg_name}"
    os.makedirs(local_dir, exist_ok=True)

    # Step 1 - Download TFRecord from GCS
    print("  Step 1: Downloading TFRecord...")
    local_path = "/content/current.tfrecord"
    url = tfrecord_path.strip()
    !gsutil cp {url} /content/current.tfrecord


    # Step 2 - Parse frames
    print("  Step 2: Parsing frames...")
    dataset = tf.data.TFRecordDataset(local_path)
    images_saved = 0
    labels_list = []



    for frame_idx, data in enumerate(dataset):
        frame = dataset_pb2.Frame()
        frame.ParseFromString(data.numpy())

        # Get FRONT camera image only
        for cam in frame.images:
            if cam.name != 1:  # 1 = FRONT camera
                continue

            # Save image
            img = Image.open(io.BytesIO(cam.image))
            img.save(f"{local_dir}/frame_{frame_idx:03d}_FRONT.jpg")
            images_saved += 1

            # Get labels for this frame
            for cam_labels in frame.camera_labels:
                if cam_labels.name != 1:
                    continue
                for label in cam_labels.labels:
                    labels_list.append({
                        'frame': frame_idx,
                        'camera': 'FRONT',
                        'type': label.type,
                        'center_x': label.box.center_x,
                        'center_y': label.box.center_y,
                        'width': label.box.width,
                        'height': label.box.length
                    })

    # Step 3 - Save labels CSV
    print("  Step 3: Saving labels...")
    df = pd.DataFrame(labels_list)
    df.to_csv(f"{local_dir}/labels.csv", index=False)

    # Step 4 - Upload to GCS
    print("  Step 4: Uploading to GCS...")
    gcs_path = f"gs://mywaymo-perdataset-2026/extracted/{seg_name}"
    local_jpg = f"{local_dir}/*.jpg"
    local_csv = f"{local_dir}/labels.csv"

    !gsutil -m -q cp {local_jpg} {gcs_path}/
    !gsutil -q cp {local_csv} {gcs_path}/labels.csv


    # Step 5 - Cleanup local files to save disk
    import shutil
    shutil.rmtree(local_dir)
    os.remove(local_path)

    print(f"  ✅ Done! {images_saved} images, {len(df)} labels saved to GCS")

print("Function ready ✅")

Function ready ✅


In [21]:
# Test with just ONE segment first
extract_one_segment(tfrecords_50[0])

  Step 1: Downloading TFRecord...
Copying gs://mywaymo-perdataset-2026/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

/ [1 files][  1.0 GiB/  1.0 GiB]   22.4 MiB/s                                   
Operation completed over 1 objects/1.0 GiB.                                      
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 198 images, 2127 labels saved to GCS


In [10]:
# Debug - print exact path
print(f"Path: '{tfrecords_50[0]}'")
print(f"Stripped: '{tfrecords_50[0].strip()}'")

# Try direct download
!gsutil cp {tfrecords_50[0].strip()} /content/current.tfrecord

# Check if file exists
import os
print(f"File exists: {os.path.exists('/content/current.tfrecord')}")

Path: 'gs://mywaymo-perdataset-2026/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord'
Stripped: 'gs://mywaymo-perdataset-2026/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord'
Copying gs://mywaymo-perdataset-2026/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

\ [1 files][  1.0 GiB/  1.0 GiB]   22.1 MiB/s                                   
Operation completed over 1 objects/1.0 GiB.                                      
File exists: True


In [ ]:
#extraction loop:
# Extract 50 segments one by one
total = len(tfrecords_50)

for i, tfrecord_path in enumerate(tfrecords_50):
    print(f"\n{'='*50}")
    print(f"Segment {i+1} of {total}")
    print(f"{'='*50}")

    try:
        extract_one_segment(tfrecord_path)
    except Exception as e:
        print(f"  ❌ Failed: {e} — skipping to next segment")
        continue

print("\n🎉 All 50 segments extracted!")


Segment 1 of 50

Processing: segment-10017090168044687777_6380_000_6400_000_wit
  Step 1: Downloading TFRecord...
CommandException: Destination URL must name a directory, bucket, or bucket
subdirectory for the multiple source form of the cp command.
  Step 2: Parsing frames...
  ❌ Failed: {{function_node __wrapped__IteratorGetNext_output_types_1_device_/job:localhost/replica:0/task:0/device:CPU:0}} /content/current.tfrecord; No such file or directory [Op:IteratorGetNext] name:  — skipping to next segment

Segment 2 of 50

Processing: segment-10023947602400723454_1120_000_1140_000_wit
  Step 1: Downloading TFRecord...
CommandException: Destination URL must name a directory, bucket, or bucket
subdirectory for the multiple source form of the cp command.
  Step 2: Parsing frames...
  ❌ Failed: {{function_node __wrapped__IteratorGetNext_output_types_1_device_/job:localhost/replica:0/task:0/device:CPU:0}} /content/current.tfrecord; No such file or directory [Op:IteratorGetNext] name:  — ski

In [22]:
# Improved extraction loop # Extract 50 segments one by one

total = len(tfrecords_50)

for i, tfrecord_path in enumerate(tfrecords_50):
    seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord-00000-of-00001', '').replace('.tfrecord', '')

    # Skip if already extracted
    check = !gsutil ls gs://mywaymo-perdataset-2026/extracted/{seg_name}/labels.csv 2>/dev/null
    if check:
        print(f"Segment {i+1} of {total} — already extracted ✅")
        continue

    print(f"\nSegment {i+1} of {total}")
    try:
        extract_one_segment(tfrecord_path)
    except Exception as e:
        print(f"  ❌ Failed: {e} — skipping")
        continue

print("\n🎉 All 50 segments done!")

Segment 1 of 50 — already extracted ✅

Segment 2 of 50
  Step 1: Downloading TFRecord...
Copying gs://mywaymo-perdataset-2026/training/segment-10023947602400723454_1120_000_1140_000_with_camera_labels.tfrecord...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

- [1 files][894.1 MiB/894.1 MiB]   22.8 MiB/s                                   
Operation completed over 1 objects/894.1 MiB.                                    
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 199 images, 8364 labels saved to GCS

Segment 3 of 50
  Step 1: Downloading TFRecord...
Copying gs://mywaymo-perdataset-2026/training/segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord...
==> NOTE: You are downloading one or more large file(s), which would
run 

In [23]:
# Check extraction results
result = !gsutil ls gs://mywaymo-perdataset-2026/extracted/
segments = [r.strip() for r in result if 'segment' in r]
print(f"Total segments in GCS: {len(segments)}")

# Check total images Each segment has 198 FRONT camera frames, so:50 segments × 198 frames = 9,900 images .
# 224,023 ÷ 50 segments = ~4,480 labels per segment on average;All 4 classes now represented — Vehicle, Pedestrian, Sign, Cyclist
!gsutil du -sh gs://mywaymo-perdataset-2026/extracted/

Total segments in GCS: 50
2.41 GiB     gs://mywaymo-perdataset-2026/extracted


In [24]:
#Training
 # Install YOLOv8
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print("YOLOv8 ready ✅")

Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Setup complete ✅ (12 CPUs, 53.0 GB RAM, 41.8/235.7 GB disk)
YOLOv8 ready ✅


In [25]:
#training pipeline:

import os
import shutil
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

# Config
GCS_EXTRACTED  = "gs://mywaymo-perdataset-2026/extracted"
GCS_MODEL_OUT  = "gs://mywaymo-perdataset-2026/models"
LOCAL_DATA_DIR = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)

CLASS_MAP    = {1: 0, 2: 1, 3: 2, 4: 3}
CLASS_NAMES  = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]
IMG_W, IMG_H = 1920, 1280
CAMERA       = "FRONT"
TRAIN_SPLIT  = 0.8
EPOCHS       = 50
IMG_SIZE     = 640
BATCH_SIZE   = 32  # L4 can handle 32
MODEL        = "yolov8m.pt"

print("Config ready ✅")
print(f"Model    : {MODEL}")
print(f"Epochs   : {EPOCHS}")
print(f"Batch    : {BATCH_SIZE}")
print(f"Classes  : {CLASS_NAMES}")

Config ready ✅
Model    : yolov8m.pt
Epochs   : 50
Batch    : 32
Classes  : ['Vehicle', 'Pedestrian', 'Sign', 'Cyclist']


In [26]:
# data download

# Download all 50 segments from GCS
raw_images_dir = LOCAL_DATA_DIR / "raw_images"
raw_labels_dir = LOCAL_DATA_DIR / "raw_labels"
raw_images_dir.mkdir(exist_ok=True)
raw_labels_dir.mkdir(exist_ok=True)

# Get segment list
result = !gsutil ls {GCS_EXTRACTED}/
segments = [r.strip().rstrip('/') for r in result if 'segment' in r]
print(f"Found {len(segments)} segments")

all_labels = []

for i, seg_path in enumerate(segments):
    seg_name = seg_path.split('/')[-1]
    print(f"Downloading segment {i+1}/{len(segments)}: {seg_name[:40]}...")

    # Download FRONT images
    !gsutil -m -q cp "{seg_path}/*_FRONT.jpg" "{raw_images_dir}/"

    # Download labels CSV
    csv_local = raw_labels_dir / f"{seg_name}_labels.csv"
    !gsutil -q cp "{seg_path}/labels.csv" "{csv_local}"

    if csv_local.exists():
        df = pd.read_csv(csv_local)
        df['segment'] = seg_name
        all_labels.append(df)

# Combine all labels
labels_df = pd.concat(all_labels, ignore_index=True)
labels_df = labels_df[labels_df['camera'] == CAMERA].reset_index(drop=True)

print(f"\nTotal images : {len(list(raw_images_dir.glob('*.jpg')))}")
print(f"Total labels : {len(labels_df)}")

Found 50 segments

Total images : 199
Total labels : 224023


Labels count is correct — 50 segments × ~4,000+ labels per segment = ~224,000 total labels ✅

Images count is wrong — 199 instead of 9,900 ⚠️

The problem is all 50 segments have frame_000_FRONT.jpg through frame_197_FRONT.jpg — same filenames! Each segment overwrites the previous one's images.

Fix — download images into segment-specific subfolders won't work for YOLO. Instead rename images with segment prefix:

In [27]:
# Clear and re-download with unique filenames
import shutil
shutil.rmtree(str(raw_images_dir))
raw_images_dir.mkdir(exist_ok=True)

all_labels = []

for i, seg_path in enumerate(segments):
    seg_name = seg_path.split('/')[-1]
    seg_short = f"seg{i:03d}"  # seg000, seg001, seg002...
    print(f"Downloading segment {i+1}/{len(segments)}...")

    # Download to temp folder first
    temp_dir = LOCAL_DATA_DIR / "temp"
    temp_dir.mkdir(exist_ok=True)
    !gsutil -m -q cp "{seg_path}/*_FRONT.jpg" "{temp_dir}/"

    # Rename with segment prefix and move
    for img in temp_dir.glob("*.jpg"):
        new_name = f"{seg_short}_{img.name}"
        img.rename(raw_images_dir / new_name)

    shutil.rmtree(str(temp_dir))

    # Download labels with segment prefix in frame column
    csv_local = raw_labels_dir / f"{seg_name}_labels.csv"
    !gsutil -q cp "{seg_path}/labels.csv" "{csv_local}"

    if csv_local.exists():
        df = pd.read_csv(csv_local)
        df['segment'] = seg_name
        df['seg_short'] = seg_short
        all_labels.append(df)

labels_df = pd.concat(all_labels, ignore_index=True)
labels_df = labels_df[labels_df['camera'] == CAMERA].reset_index(drop=True)

print(f"\nTotal images : {len(list(raw_images_dir.glob('*.jpg')))}")
print(f"Total labels : {len(labels_df)}")


Total images : 9919
Total labels : 224023


In [28]:
#label conversion cell:
def get_image_size(img_path):
    try:
        with Image.open(img_path) as img:
            return img.size
    except:
        return IMG_W, IMG_H

def convert_to_yolo(center_x, center_y, width, height, img_w, img_h):
    cx = max(0.0, min(1.0, center_x / img_w))
    cy = max(0.0, min(1.0, center_y / img_h))
    w  = max(0.001, min(1.0, width / img_w))
    h  = max(0.001, min(1.0, height / img_h))
    return cx, cy, w, h

# Create YOLO label files
yolo_labels_dir = LOCAL_DATA_DIR / "yolo_labels"
yolo_labels_dir.mkdir(exist_ok=True)

skipped = 0
processed = 0

for (seg_short, frame), group in labels_df.groupby(['seg_short', 'frame']):
    img_filename = f"{seg_short}_frame_{int(frame):03d}_FRONT.jpg"
    img_path = raw_images_dir / img_filename

    if not img_path.exists():
        skipped += 1
        continue

    img_w, img_h = get_image_size(img_path)
    label_path = yolo_labels_dir / img_filename.replace('.jpg', '.txt')

    with open(label_path, 'w') as f:
        for _, row in group.iterrows():
            yolo_class = CLASS_MAP.get(int(row['type']))
            if yolo_class is None:
                continue
            cx, cy, w, h = convert_to_yolo(
                row['center_x'], row['center_y'],
                row['width'], row['height'],
                img_w, img_h
            )
            f.write(f"{yolo_class} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

    processed += 1

print(f"YOLO labels created : {processed}")
print(f"Skipped             : {skipped}")

YOLO labels created : 9717
Skipped             : 0


9,717 YOLO label files created — slightly less than 9,919 images because some frames have no objects (empty scenes with no vehicles/pedestrians/signs/cyclists).
That's completely normal and expected. ✅

In [29]:
# Create train/val 80/20 split,can take 10% from the remaining 748 segments as a completely unseen test set
images_dir  = LOCAL_DATA_DIR / "images"
labels_dir  = LOCAL_DATA_DIR / "labels"

(images_dir / "train").mkdir(parents=True, exist_ok=True)
(images_dir / "val").mkdir(parents=True, exist_ok=True)
(labels_dir / "train").mkdir(parents=True, exist_ok=True)
(labels_dir / "val").mkdir(parents=True, exist_ok=True)

# Only use images that have labels
labeled_images = [f for f in raw_images_dir.glob("*.jpg")
                  if (yolo_labels_dir / f.name.replace('.jpg', '.txt')).exists()]

# Shuffle and split
random.seed(42)
random.shuffle(labeled_images)
split_idx = int(len(labeled_images) * TRAIN_SPLIT)
train_imgs = labeled_images[:split_idx]
val_imgs   = labeled_images[split_idx:]

# Copy to train/val folders
for img in train_imgs:
    shutil.copy(img, images_dir / "train" / img.name)
    shutil.copy(yolo_labels_dir / img.name.replace('.jpg', '.txt'),
                labels_dir / "train" / img.name.replace('.jpg', '.txt'))

for img in val_imgs:
    shutil.copy(img, images_dir / "val" / img.name)
    shutil.copy(yolo_labels_dir / img.name.replace('.jpg', '.txt'),
                labels_dir / "val" / img.name.replace('.jpg', '.txt'))

print(f"Train images : {len(train_imgs)}")
print(f"Val images   : {len(val_imgs)}")

Train images : 7773
Val images   : 1944


AML is a simple configuration file format — human readable, like a settings file.

What's in our YAML:

path: /content/waymo_yolo        # where dataset lives

train: images/train              # train images folder

val: images/val                  # val images folder

nc: 4                            # number of classes

names: ['Vehicle', 'Pedestrian', 'Sign', 'Cyclist']  # class names

YOLOv8 need YAML file
When you call model.train() you just pass one file: model.train(data="waymo.yaml")


In [30]:
#add the dataset YAML config cell:

# Create dataset YAML file
yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: images/train
val: images/val

nc: 4
names: {CLASS_NAMES}
"""

yaml_path = LOCAL_DATA_DIR / "waymo.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("Dataset YAML created ✅")
print(yaml_content)

Dataset YAML created ✅

path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names: ['Vehicle', 'Pedestrian', 'Sign', 'Cyclist']



In [31]:
#Final training

# Train YOLOv8m
model = YOLO(MODEL)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name="waymo_yolov8m",
    project="/content/runs",
    device=0,
    workers=4,
    patience=10,
    save=True,
    plots=True,
    verbose=True
)

print("Training complete ✅")

Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/waymo_yolo/waymo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=waymo_yolov8m, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, persp

In [32]:
# Save model to GCS
!gsutil cp /content/runs/waymo_yolov8m/weights/best.pt gs://mywaymo-perdataset-2026/models/waymo_yolov8m_best.pt
!gsutil cp /content/runs/waymo_yolov8m/weights/last.pt gs://mywaymo-perdataset-2026/models/waymo_yolov8m_last.pt
!gsutil -m cp /content/runs/waymo_yolov8m/*.png gs://mywaymo-perdataset-2026/models/
print("Model saved to GCS ✅")

Copying file:///content/runs/waymo_yolov8m/weights/best.pt [Content-Type=application/vnd.snesdev-page-table]...
|
Operation completed over 1 objects/49.6 MiB.                                     
Copying file:///content/runs/waymo_yolov8m/weights/last.pt [Content-Type=application/vnd.snesdev-page-table]...
\ [1 files][ 49.6 MiB/ 49.6 MiB]                                                
Operation completed over 1 objects/49.6 MiB.                                     
Copying file:///content/runs/waymo_yolov8m/BoxF1_curve.png [Content-Type=image/png]...
Copying file:///content/runs/waymo_yolov8m/BoxP_curve.png [Content-Type=image/png]...
Copying file:///content/runs/waymo_yolov8m/BoxPR_curve.png [Content-Type=image/png]...
Copying file:///content/runs/waymo_yolov8m/BoxR_curve.png [Content-Type=image/png]...
Copying file:///content/runs/waymo_yolov8m/confusion_matrix_normalized.png [Content-Type=image/png]...
Copying file:///content/runs/waymo_yolov8m/confusion_matrix.png [Content-Type=im